# 05 — Evaluación del Sistema de Recomendación
## Cross-Selling Recommender | Proyecto Final Henry

**Split temporal (correcto estadísticamente):**
- Train: `order_products__prior.csv` — historial completo
- Test: `order_products__train.csv` — última orden real por usuario (ground truth)

**Métricas:** Precision@K, Recall@K, NDCG@K, Hit Rate@K, Coverage


In [ ]:
# ── 1. Configuración ────────────────────────────────────────────────────────
import sys, logging
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

PROJECT_ROOT = Path().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import load_config, load_instacart_tables, load_train_split, build_master_dataframe
from src.data.preprocessor import run_cleaning_pipeline
from src.models.association_rules import load_rules, get_recommendations_for_product
from src.models.collaborative_filter import load_als_model, build_implicit_matrix, get_recommendations_for_user
from src.evaluation.metrics import evaluate_model, coverage

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s', datefmt='%H:%M:%S')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120})
FIGURES_DIR = PROJECT_ROOT / 'outputs' / 'figures'
REPORTS_DIR = PROJECT_ROOT / 'outputs' / 'reports'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
K_VALUES = [5, 10]
np.random.seed(RANDOM_STATE)
print(f'PROJECT_ROOT: {PROJECT_ROOT}')


---
## 2. Carga de Datos y Modelos

In [ ]:
# ── 2.1 Dataset ─────────────────────────────────────────────────────────────
cfg = load_config()
orders, op_prior, products, aisles, departments = load_instacart_tables(cfg)
df_train_split = load_train_split(cfg)

df_master = build_master_dataframe(
    orders=orders, order_products=op_prior,
    products=products, aisles=aisles,
    departments=departments, eval_set='prior',
)
product_names = products.set_index('product_id')['product_name'].to_dict()
catalog = products['product_id'].tolist()

print(f'Historial (prior)    : {len(df_master):,} ítems')
print(f'Ground truth (train) : {len(df_train_split):,} ítems')


In [ ]:
# ── 2.2 Pipeline de limpieza ─────────────────────────────────────────────────
df_full, df_apriori = run_cleaning_pipeline(
    df=df_master,
    min_basket_items=2, add_temporal=True,
    apriori_sample=200_000, random_state=RANDOM_STATE,
)
print(f'df_full   : {df_full.shape}')
print(f'Usuarios  : {df_full["user_id"].nunique():,}')


In [ ]:
# ── 2.3 Cargar modelos entrenados ─────────────────────────────────────────────
rules = load_rules(filename='fpgrowth_rules.pkl')
als_model, user_ids, product_ids = load_als_model(filename='als_model.pkl')
print(f'FP-Growth: {len(rules):,} reglas')
print(f'ALS      : {len(user_ids):,} usuarios | {len(product_ids):,} productos')


In [ ]:
# ── 2.4 Reconstruir matriz ALS para inferencia ────────────────────────────────
item_user_matrix, _, _ = build_implicit_matrix(df=df_full, alpha=40.0)
print(f'Matriz ALS: {item_user_matrix.shape}')


---
## 3. Ground Truth (Split Temporal)

In [ ]:
# ── 3.1 Construir ground truth: última orden por usuario ──────────────────────
# Unir df_train_split (product_ids) con orders para obtener user_id
train_orders = orders[orders['eval_set'] == 'train'][['order_id','user_id']]
gt_df = df_train_split.merge(train_orders, on='order_id', how='inner')
ground_truth = gt_df.groupby('user_id')['product_id'].apply(list).to_dict()

print(f'Usuarios con ground truth: {len(ground_truth):,}')
gt_sizes = [len(v) for v in ground_truth.values()]
print(f'Ítems/orden media        : {np.mean(gt_sizes):.1f}')
print(f'Ítems/orden mediana      : {np.median(gt_sizes):.0f}')


In [ ]:
# ── 3.2 Muestra de evaluación — usuarios en ambos modelos ────────────────────
eval_users = list(set(user_ids) & set(ground_truth.keys()))
N_EVAL = min(2000, len(eval_users))
rng = np.random.default_rng(RANDOM_STATE)
eval_sample = rng.choice(eval_users, size=N_EVAL, replace=False).tolist()
print(f'Usuarios evaluables: {len(eval_users):,}')
print(f'Muestra de evaluación: {N_EVAL:,}')


---
## 4. Generación de Recomendaciones

In [ ]:
# ── 4.1 Recomendaciones ALS ──────────────────────────────────────────────────
print(f'Generando recomendaciones ALS para {N_EVAL:,} usuarios...')
recs_als = {}
for uid in eval_sample:
    r = get_recommendations_for_user(
        user_id=uid, model=als_model, user_ids=user_ids, product_ids=product_ids,
        item_user_matrix=item_user_matrix, top_k=max(K_VALUES),
    )
    if not r.empty:
        recs_als[uid] = r['product_id'].tolist()
print(f'Recomendaciones ALS generadas: {len(recs_als):,} usuarios')


In [ ]:
# ── 4.2 Recomendaciones FP-Growth (product-based) ────────────────────────────
# Estrategia: para cada usuario tomar su producto más frecuente del historial
# y recomendar los consecuentes de las reglas correspondientes.
from collections import Counter
user_history = df_full.groupby('user_id')['product_id'].apply(list).to_dict()

print(f'Generando recomendaciones FP-Growth para {N_EVAL:,} usuarios...')
recs_ar = {}
for uid in eval_sample:
    hist = user_history.get(uid, [])
    if not hist:
        continue
    top_prod = Counter(hist).most_common(1)[0][0]
    r = get_recommendations_for_product(product_id=top_prod, rules=rules, top_k=max(K_VALUES))
    if not r.empty:
        recs_ar[uid] = r['product_id'].tolist()
print(f'Recomendaciones FP-Growth generadas: {len(recs_ar):,} usuarios')


---
## 5. Cálculo de Métricas

In [ ]:
# ── 5.1 Evaluar ALS ──────────────────────────────────────────────────────────
gt_als = {u: ground_truth[u] for u in recs_als if u in ground_truth}
print('EVALUACIÓN — ALS')
results_als = evaluate_model(recommendations=recs_als, ground_truth=gt_als,
                              catalog=catalog, k_values=K_VALUES)
display(results_als.round(4))


In [ ]:
# ── 5.2 Evaluar FP-Growth ────────────────────────────────────────────────────
gt_ar = {u: ground_truth[u] for u in recs_ar if u in ground_truth}
print('EVALUACIÓN — FP-Growth')
results_ar = evaluate_model(recommendations=recs_ar, ground_truth=gt_ar,
                             catalog=catalog, k_values=K_VALUES)
display(results_ar.round(4))


In [ ]:
# ── 5.3 Tabla comparativa ─────────────────────────────────────────────────────
def add_model(df, name):
    d = df.reset_index()
    d.insert(0, 'Modelo', name)
    return d

comparison = pd.concat([add_model(results_als, 'ALS'), add_model(results_ar, 'FP-Growth')],
                        ignore_index=True)
print('TABLA COMPARATIVA:')
display(comparison.round(4))


---
## 6. Visualización Comparativa

In [ ]:
# ── 6.1 Barras comparativas por métrica y K ──────────────────────────────────
metrics_to_plot = ['Precision@K', 'Recall@K', 'NDCG@K', 'HitRate@K']
colors = {'ALS': '#4C72B0', 'FP-Growth': '#DD8452'}

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('Comparativa de modelos — Métricas de Recomendación', fontsize=13, fontweight='bold')

for ax, metric in zip(axes.flatten(), metrics_to_plot):
    for model_name in ['ALS', 'FP-Growth']:
        sub = comparison[comparison['Modelo'] == model_name]
        ax.bar([f'K={k}' for k in sub['K']], sub[metric],
               color=colors[model_name], alpha=0.85, label=model_name,
               width=0.35 if model_name == 'ALS' else -0.35, align='edge')
    ax.set_title(metric); ax.set_ylabel(metric); ax.legend(fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))

plt.tight_layout()
fig.savefig(FIGURES_DIR / '14_evaluation_comparison.png', bbox_inches='tight')
plt.show()
print('Guardado: outputs/figures/14_evaluation_comparison.png')


---
## 7. Impacto en Basket Size (KPI de Negocio)

In [ ]:
# ── 7.1 Basket size actual (ground truth) ────────────────────────────────────
actual_sizes = [len(v) for v in ground_truth.values()]
baseline     = np.mean(actual_sizes)

# HR@10 como proxy de probabilidad de al menos 1 hit
def get_hr(df_comp, model_name, k=10):
    row = df_comp[(df_comp['Modelo'] == model_name) & (df_comp['K'] == k)]
    return row['HitRate@K'].values[0] if len(row) > 0 else 0.0

hr_als = get_hr(comparison, 'ALS')
hr_ar  = get_hr(comparison, 'FP-Growth')
ACCEPTANCE = 0.10   # tasa de aceptación conservadora (benchmark e-commerce)

basket_als = baseline + ACCEPTANCE * hr_als
basket_ar  = baseline + ACCEPTANCE * hr_ar

print(f'Basket size baseline   : {baseline:.2f}')
print(f'ALS     HR@10={hr_als:.3f} → Basket proyectado: {basket_als:.3f} (+{basket_als-baseline:.4f})')
print(f'FP-Growth HR@10={hr_ar:.3f} → Basket proyectado: {basket_ar:.3f} (+{basket_ar-baseline:.4f})')
print(f'Supuesto: acceptance_rate = {ACCEPTANCE:.0%}')


In [ ]:
# ── 7.2 Gráfico de impacto ────────────────────────────────────────────────────
models  = ['Baseline', 'ALS', 'FP-Growth']
baskets = [baseline, basket_als, basket_ar]
bcolors = ['#AAAAAA', '#4C72B0', '#DD8452']

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(models, baskets, color=bcolors, alpha=0.9, edgecolor='white', width=0.5)
for bar, val, m in zip(bars, baskets, models):
    label = f'{val:.3f}' + (f'\n(+{val-baseline:.4f})' if m != 'Baseline' else '')
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            label, ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('Basket Size (ítems/orden)')
ax.set_title(f'Impacto en Basket Size (acceptance={ACCEPTANCE:.0%})', fontsize=12)
ax.set_ylim(0, max(baskets) * 1.15)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '17_eval_basket_size_impact.png', bbox_inches='tight')
plt.show()


---
## 8. Cobertura de Catálogo

In [ ]:
# ── 8.1 Cobertura por modelo ─────────────────────────────────────────────────
cov_als = coverage(recs_als, catalog)
cov_ar  = coverage(recs_ar,  catalog)
print(f'Catálogo total     : {len(catalog):,}')
print(f'ALS Coverage       : {cov_als:.4%}')
print(f'FP-Growth Coverage : {cov_ar:.4%}')
print()
print('FP-Growth < ALS porque opera solo sobre productos con soporte >= 1%.')
print('ALS cubre cualquier producto que haya aparecido en el historial.')


---
## 9. Resumen Final

In [ ]:
# ── 9.1 Tabla final @ K=10 ───────────────────────────────────────────────────
final = comparison[comparison['K'] == 10].set_index('Modelo').drop(columns=['K'])
print('RESULTADOS FINALES @ K=10')
display(final.round(4))


In [ ]:
# ── 9.2 Guardar resultados ────────────────────────────────────────────────────
comparison.to_csv(REPORTS_DIR / 'evaluation_results.csv', index=False)
print('Guardado: outputs/reports/evaluation_results.csv')

print()
print('=== CONCLUSIONES ===')
print('FP-Growth: reglas interpretables, alta precisión, baja cobertura (long tail excluida).')
print('ALS: recomendaciones personalizadas, cubre long tail, mayor cobertura.')
print('Ambos son complementarios: AR para páginas de producto, ALS para homepage.')
print('Con acceptance_rate=10%, ambos proyectan mejora en basket size.')
